# TAHAP 05 — Program Studi Recommender System & Roadmap Mapping

## EduPath Career Assistant

Pada tahap ini dilakukan pembangunan sistem rekomendasi program studi S1 berbasis content-based recommender system. Sistem ini memanfaatkan profil teks program studi, minat user, mata pelajaran favorit, hobi, gaya belajar, dan tujuan karier untuk menghasilkan rekomendasi Top-N program studi.

Pendekatan utama yang digunakan adalah:

1. Text preprocessing sederhana
2. TF-IDF Vectorization
3. Cosine Similarity
4. Rule-based structured scoring
5. Roadmap mapping
6. Career mapping
7. Skill mapping

Tahap ini menjadi penghubung antara hasil intent classification pada Tahap 04 dengan fitur chatbot rekomendasi pada tahap berikutnya.

In [1]:
# ============================================================
# TAHAP 05 — Import Library
# ============================================================

from pathlib import Path
import re
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Library berhasil di-load.")

Library berhasil di-load.


In [2]:
# ============================================================
# Setup Path Project
# ============================================================

PROJECT_ROOT = Path.cwd()

# Jika notebook dijalankan dari folder notebooks, naik satu level ke root project
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports" / "stage05"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       :", PROJECT_ROOT)
print("DATA_PROCESSED_DIR :", DATA_PROCESSED_DIR)
print("MODELS_DIR         :", MODELS_DIR)
print("REPORTS_DIR        :", REPORTS_DIR)

PROJECT_ROOT       : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant
DATA_PROCESSED_DIR : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\processed
MODELS_DIR         : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\models
REPORTS_DIR        : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage05


In [3]:
# ============================================================
# Load Dataset Processed
# ============================================================

intent_path = DATA_PROCESSED_DIR / "intent_dataset_processed.csv"
program_path = DATA_PROCESSED_DIR / "program_studi_s1_processed.csv"
career_path = DATA_PROCESSED_DIR / "karier_processed.csv"
skill_path = DATA_PROCESSED_DIR / "skill_awal_processed.csv"
roadmap_path = DATA_PROCESSED_DIR / "roadmap_belajar_processed.csv"
normalisasi_path = DATA_PROCESSED_DIR / "normalisasi_bahasa_processed.csv"

required_files = [
    intent_path,
    program_path,
    career_path,
    skill_path,
    roadmap_path,
    normalisasi_path
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}")

intent_df = pd.read_csv(intent_path, encoding="utf-8-sig")
program_df = pd.read_csv(program_path, encoding="utf-8-sig")
career_df = pd.read_csv(career_path, encoding="utf-8-sig")
skill_df = pd.read_csv(skill_path, encoding="utf-8-sig")
roadmap_df = pd.read_csv(roadmap_path, encoding="utf-8-sig")
normalisasi_df = pd.read_csv(normalisasi_path, encoding="utf-8-sig")

print("Dataset berhasil di-load.")
print("Intent dataset       :", intent_df.shape)
print("Program studi        :", program_df.shape)
print("Karier               :", career_df.shape)
print("Skill awal           :", skill_df.shape)
print("Roadmap belajar      :", roadmap_df.shape)
print("Normalisasi bahasa   :", normalisasi_df.shape)

Dataset berhasil di-load.
Intent dataset       : (12, 21)
Program studi        : (5, 22)
Karier               : (5, 15)
Skill awal           : (7, 15)
Roadmap belajar      : (6, 15)
Normalisasi bahasa   : (18, 10)


In [4]:
# ============================================================
# Validasi Kolom Penting
# ============================================================

required_program_cols = [
    "program_id",
    "nama_program_studi",
    "jenjang",
    "rumpun_ilmu",
    "bidang_keilmuan",
    "deskripsi_singkat",
    "minat_cocok",
    "mapel_relevan",
    "hobi_relevan",
    "gaya_belajar_cocok",
    "karakteristik_cocok",
    "prospek_karier_id_list",
    "skill_id_list",
    "keyword_rekomendasi",
    "program_profile_text"
]

required_career_cols = [
    "karier_id",
    "nama_karier",
    "bidang_karier",
    "deskripsi_singkat",
    "keyword_karier",
    "career_profile_text"
]

required_skill_cols = [
    "skill_id",
    "nama_skill",
    "kategori_skill",
    "deskripsi_singkat",
    "keyword_skill",
    "skill_profile_text"
]

required_roadmap_cols = [
    "roadmap_id",
    "program_id",
    "fase",
    "urutan_fase",
    "durasi_rekomendasi",
    "tujuan_fase",
    "materi_pokok",
    "aktivitas_praktik",
    "output_portofolio",
    "tools_umum",
    "indikator_capaian"
]

def validate_columns(df, required_cols, df_name):
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Kolom berikut belum ada pada {df_name}: {missing_cols}")
    print(f"Validasi kolom {df_name}: OK")

validate_columns(program_df, required_program_cols, "program_df")
validate_columns(career_df, required_career_cols, "career_df")
validate_columns(skill_df, required_skill_cols, "skill_df")
validate_columns(roadmap_df, required_roadmap_cols, "roadmap_df")

Validasi kolom program_df: OK
Validasi kolom career_df: OK
Validasi kolom skill_df: OK
Validasi kolom roadmap_df: OK


## Desain Metode Rekomendasi

Sistem rekomendasi pada tahap ini menggunakan pendekatan **content-based recommender system**.

Setiap program studi direpresentasikan sebagai profil teks yang berisi:

- nama program studi
- rumpun ilmu
- bidang keilmuan
- deskripsi program studi
- minat yang cocok
- mata pelajaran relevan
- hobi relevan
- gaya belajar yang cocok
- karakteristik siswa yang cocok
- keyword rekomendasi
- prospek karier terkait

Kemudian input user diubah menjadi profil teks sederhana. Profil user tersebut dibandingkan dengan profil program studi menggunakan:

\[
Cosine Similarity = \frac{A \cdot B}{||A|| \times ||B||}
\]

Selain similarity berbasis teks, sistem juga menambahkan skor berbasis aturan sederhana untuk mempertimbangkan:

1. minat
2. mata pelajaran favorit
3. hobi
4. gaya belajar
5. tujuan karier

Skor akhir dihitung dengan formula:

\[
Final Score = 0.65 \times Text Similarity + 0.35 \times Structured Score
\]

Formula ini dipilih agar rekomendasi tetap berbasis teks, tetapi masih mempertimbangkan atribut profil user secara eksplisit.

In [5]:
# ============================================================
# Fungsi Helper Preprocessing
# ============================================================

def split_pipe(value):
    """
    Memecah teks dengan delimiter pipe '|'.
    Contoh: 'data|statistik|analisis' -> ['data', 'statistik', 'analisis']
    """
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [item.strip() for item in str(value).split("|") if item.strip()]


# Membuat dictionary normalisasi dari dataset normalisasi bahasa
normalisasi_map = dict(
    zip(
        normalisasi_df["kata_input_clean"].astype(str),
        normalisasi_df["kata_baku_clean"].astype(str)
    )
)

# Tambahan normalisasi manual agar sistem lebih robust
additional_normalization = {
    "ngoding": "pemrograman",
    "coding": "pemrograman",
    "programming": "pemrograman",
    "prodi": "program studi",
    "jurusan": "program studi",
    "kuliah": "program studi",
    "kerja": "karier",
    "karir": "karier",
    "pengen": "ingin",
    "pingin": "ingin",
    "seneng": "suka",
    "resep": "suka",
    "dadi": "menjadi",
    "naon": "apa",
    "jeung": "dan",
    "nu": "yang"
}

normalisasi_map.update(additional_normalization)

# Stopwords sederhana
STOPWORDS = set("""
yang dan di ke dari untuk dengan atau saya aku gue gw abdi ingin mau pengin pingin kalau jika itu ini apa apakah ya kak dulu bidang cocok pas masuk harus belajar menjadi
""".split())


def clean_text(text):
    """
    Membersihkan teks dari karakter non-alfanumerik.
    """
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s|/+-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_text(text):
    """
    Melakukan normalisasi kata sederhana berdasarkan kamus normalisasi.
    """
    text = clean_text(text)
    tokens = text.split()

    normalized_tokens = []
    for token in tokens:
        replacement = normalisasi_map.get(token, token)
        normalized_tokens.extend(str(replacement).split())

    return " ".join(normalized_tokens)


def preprocess_text(text):
    """
    Membersihkan, menormalisasi, dan menghapus stopwords sederhana.
    """
    text = normalize_text(text)
    tokens = [
        token for token in text.split()
        if token not in STOPWORDS and len(token) > 1
    ]
    return " ".join(tokens)


def token_overlap_score(query, target):
    """
    Menghitung skor irisan token antara query dan target.
    Nilai berada pada rentang 0 sampai 1.
    """
    query_tokens = set(preprocess_text(query).split())
    target_tokens = set(preprocess_text(target).split())

    if not query_tokens:
        return 0.0

    return len(query_tokens.intersection(target_tokens)) / len(query_tokens)


print("Fungsi preprocessing berhasil dibuat.")

Fungsi preprocessing berhasil dibuat.


In [6]:
# ============================================================
# Membuat Lookup Karier dan Skill
# ============================================================

career_lookup = career_df.set_index("karier_id").to_dict("index")
skill_lookup = skill_df.set_index("skill_id").to_dict("index")


def mapped_career_text(career_id_list):
    """
    Mengambil profil karier berdasarkan daftar ID karier pada program studi.
    """
    texts = []

    for career_id in split_pipe(career_id_list):
        if career_id in career_lookup:
            career = career_lookup[career_id]
            texts.append(
                " ".join([
                    str(career.get("nama_karier", "")),
                    str(career.get("bidang_karier", "")),
                    str(career.get("keyword_karier", "")),
                    str(career.get("career_profile_text", ""))
                ])
            )

    return " ".join(texts)


def mapped_career_names(career_id_list):
    """
    Mengambil nama karier berdasarkan daftar ID karier.
    """
    names = []

    for career_id in split_pipe(career_id_list):
        if career_id in career_lookup:
            names.append(career_lookup[career_id].get("nama_karier", ""))

    return ", ".join([name for name in names if name])


def mapped_skill_names(skill_id_list):
    """
    Mengambil nama skill berdasarkan daftar ID skill.
    """
    names = []

    for skill_id in split_pipe(skill_id_list):
        if skill_id in skill_lookup:
            names.append(skill_lookup[skill_id].get("nama_skill", ""))

    return ", ".join([name for name in names if name])


program_df["mapped_career_text"] = program_df["prospek_karier_id_list"].apply(mapped_career_text)
program_df["mapped_career_names"] = program_df["prospek_karier_id_list"].apply(mapped_career_names)
program_df["mapped_skill_names"] = program_df["skill_id_list"].apply(mapped_skill_names)

display(program_df[[
    "program_id",
    "nama_program_studi",
    "mapped_career_names",
    "mapped_skill_names"
]])

,program_id,nama_program_studi,mapped_career_names,mapped_skill_names
0,PRG001,Teknik Informatika,Software Engineer,"Dasar Pemrograman Python, Dasar Basis Data dan SQL, Algoritma dan Struktur Data Dasar"
1,PRG002,Sistem Informasi,"Business Analyst, Data Analyst","Dasar Basis Data dan SQL, Analisis Data Dasar, Komunikasi dan Analisis Kebutuhan"
2,PRG003,Sains Data,"Data Analyst, Data Scientist","Dasar Pemrograman Python, Dasar Basis Data dan SQL, Analisis Data Dasar, Statistika Dasar"
3,PRG004,Statistika,"Data Analyst, Data Scientist","Analisis Data Dasar, Statistika Dasar"
4,PRG005,Desain Komunikasi Visual,UI/UX Designer,Dasar Desain Visual


In [7]:
# ============================================================
# Membuat Corpus dan TF-IDF Vectorizer
# ============================================================

# Corpus dibuat dari beberapa sumber agar vocabulary lebih kaya
corpus = pd.concat([
    program_df["program_profile_text"].fillna(""),
    program_df["mapped_career_text"].fillna(""),
    career_df["career_profile_text"].fillna(""),
    skill_df["skill_profile_text"].fillna(""),
    intent_df["utterance_preprocessed"].fillna("")
], ignore_index=True).tolist()

program_profile_for_model = (
    program_df["program_profile_text"].fillna("") + " " +
    program_df["mapped_career_text"].fillna("") + " " +
    program_df["mapped_skill_names"].fillna("")
)

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

tfidf_vectorizer.fit(corpus)

program_matrix = tfidf_vectorizer.transform(program_profile_for_model)

print("TF-IDF Vectorizer berhasil dibuat.")
print("Jumlah dokumen corpus:", len(corpus))
print("Jumlah fitur TF-IDF :", len(tfidf_vectorizer.get_feature_names_out()))
print("Ukuran matrix program:", program_matrix.shape)

TF-IDF Vectorizer berhasil dibuat.
Jumlah dokumen corpus: 34
Jumlah fitur TF-IDF : 795
Ukuran matrix program: (5, 795)


In [8]:
# ============================================================
# Fungsi Membentuk User Profile
# ============================================================

def build_user_profile_text(
    input_text,
    minat="",
    mapel="",
    hobi="",
    gaya_belajar="",
    tujuan_karier=""
):
    """
    Membentuk profil user dari teks input dan atribut tambahan.
    """
    raw_profile = " ".join([
        str(input_text),
        str(minat),
        str(mapel),
        str(hobi),
        str(gaya_belajar),
        str(tujuan_karier)
    ])

    return preprocess_text(raw_profile)


sample_user_profile = build_user_profile_text(
    input_text="Saya suka matematika, statistik, data, dan ingin menjadi data analyst",
    mapel="Matematika|Informatika",
    hobi="menganalisis data|membuat grafik",
    gaya_belajar="analitis|praktik",
    tujuan_karier="data analyst"
)

print("Contoh user profile:")
print(sample_user_profile)

Contoh user profile:
suka matematika statistik data data analyst matematika|informatika menganalisis data|membuat grafik analitis|praktik data analyst


In [9]:
# ============================================================
# Fungsi Program Studi Recommender System
# ============================================================

def recommend_programs(
    input_text,
    minat="",
    mapel="",
    hobi="",
    gaya_belajar="",
    tujuan_karier="",
    top_n=3
):
    """
    Memberikan rekomendasi Top-N program studi berdasarkan:
    1. TF-IDF + Cosine Similarity
    2. Rule-based structured score
    3. Mapping karier dan skill awal
    """

    user_profile_text = build_user_profile_text(
        input_text=input_text,
        minat=minat,
        mapel=mapel,
        hobi=hobi,
        gaya_belajar=gaya_belajar,
        tujuan_karier=tujuan_karier
    )

    user_vector = tfidf_vectorizer.transform([user_profile_text])
    text_similarities = cosine_similarity(user_vector, program_matrix).flatten()

    recommendation_rows = []

    for idx, row in program_df.iterrows():

        # Score berbasis minat dan keyword program studi
        minat_score = token_overlap_score(
            " ".join([input_text, minat]),
            " ".join([
                str(row["minat_cocok"]),
                str(row["keyword_rekomendasi"]),
                str(row["program_profile_text"])
            ])
        )

        # Score berbasis mapel
        mapel_score = token_overlap_score(
            mapel,
            row["mapel_relevan"]
        )

        # Score berbasis hobi
        hobi_score = token_overlap_score(
            hobi,
            row["hobi_relevan"]
        )

        # Score berbasis gaya belajar
        gaya_score = token_overlap_score(
            gaya_belajar,
            row["gaya_belajar_cocok"]
        )

        # Score berbasis tujuan karier
        career_score = token_overlap_score(
            " ".join([input_text, tujuan_karier]),
            row["mapped_career_text"]
        )

        # Komponen structured score
        components = []
        weights = []

        # Input text selalu digunakan untuk minat dan career
        components.append(minat_score)
        weights.append(0.40)

        components.append(career_score)
        weights.append(0.25)

        if str(mapel).strip():
            components.append(mapel_score)
            weights.append(0.15)

        if str(hobi).strip():
            components.append(hobi_score)
            weights.append(0.10)

        if str(gaya_belajar).strip():
            components.append(gaya_score)
            weights.append(0.10)

        structured_score = np.average(components, weights=weights)

        text_similarity = text_similarities[idx]

        final_score = (0.65 * text_similarity) + (0.35 * structured_score)

        # Keyword yang cocok untuk alasan rekomendasi
        program_text = " ".join([
            str(row["program_profile_text"]),
            str(row["keyword_rekomendasi"]),
            str(row["mapped_career_text"]),
            str(row["mapped_skill_names"])
        ])

        user_tokens = set(user_profile_text.split())
        program_tokens = set(preprocess_text(program_text).split())

        matched_keywords = sorted(list(user_tokens.intersection(program_tokens)))

        if matched_keywords:
            alasan = (
                "Cocok karena profil user memiliki kesesuaian pada kata kunci: "
                + ", ".join(matched_keywords[:8])
            )
        else:
            alasan = (
                "Cocok berdasarkan kemiripan umum profil teks, namun kata kunci eksplisit masih terbatas."
            )

        recommendation_rows.append({
            "rank": None,
            "program_id": row["program_id"],
            "nama_program_studi": row["nama_program_studi"],
            "jenjang": row["jenjang"],
            "rumpun_ilmu": row["rumpun_ilmu"],
            "bidang_keilmuan": row["bidang_keilmuan"],
            "text_similarity": round(float(text_similarity), 4),
            "structured_score": round(float(structured_score), 4),
            "final_score": round(float(final_score), 4),
            "final_score_percent": round(float(final_score) * 100, 2),
            "matched_keywords": ", ".join(matched_keywords[:12]),
            "prospek_karier": row["mapped_career_names"],
            "skill_awal": row["mapped_skill_names"],
            "alasan_rekomendasi": alasan
        })

    result_df = (
        pd.DataFrame(recommendation_rows)
        .sort_values("final_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    result_df["rank"] = range(1, len(result_df) + 1)

    return result_df


print("Fungsi recommend_programs berhasil dibuat.")

Fungsi recommend_programs berhasil dibuat.


In [10]:
# ============================================================
# Fungsi Roadmap Mapping
# ============================================================

def get_roadmap_by_program(program_id):
    """
    Mengambil roadmap belajar berdasarkan program_id.
    """
    roadmap_result = (
        roadmap_df[roadmap_df["program_id"] == program_id]
        .sort_values("urutan_fase")
        .reset_index(drop=True)
    )

    selected_cols = [
        "program_id",
        "fase",
        "urutan_fase",
        "durasi_rekomendasi",
        "tujuan_fase",
        "materi_pokok",
        "aktivitas_praktik",
        "output_portofolio",
        "tools_umum",
        "indikator_capaian"
    ]

    return roadmap_result[selected_cols]


def show_recommendation_with_roadmap(recommendation_df):
    """
    Menampilkan rekomendasi program studi beserta roadmap untuk program terbaik.
    """
    if recommendation_df.empty:
        print("Tidak ada rekomendasi yang dapat ditampilkan.")
        return None, None

    top_program_id = recommendation_df.iloc[0]["program_id"]
    top_program_name = recommendation_df.iloc[0]["nama_program_studi"]

    roadmap_result = get_roadmap_by_program(top_program_id)

    print("=" * 80)
    print("TOP REKOMENDASI PROGRAM STUDI")
    print("=" * 80)
    display(recommendation_df)

    print("\n" + "=" * 80)
    print(f"ROADMAP BELAJAR UNTUK: {top_program_name} ({top_program_id})")
    print("=" * 80)
    display(roadmap_result)

    return recommendation_df, roadmap_result


print("Fungsi roadmap mapping berhasil dibuat.")

Fungsi roadmap mapping berhasil dibuat.


In [11]:
# ============================================================
# Uji Coba Rekomendasi 1
# ============================================================

test_recommendation_1 = recommend_programs(
    input_text="Saya suka matematika, statistik, data, membuat grafik, dan ingin menjadi data analyst",
    minat="data|statistik|analisis",
    mapel="Matematika|Informatika",
    hobi="menganalisis data|membuat grafik",
    gaya_belajar="analitis|praktik",
    tujuan_karier="data analyst",
    top_n=3
)

rec_result_1, roadmap_result_1 = show_recommendation_with_roadmap(test_recommendation_1)

TOP REKOMENDASI PROGRAM STUDI


,rank,program_id,nama_program_studi,jenjang,rumpun_ilmu,bidang_keilmuan,text_similarity,structured_score,final_score,final_score_percent,matched_keywords,prospek_karier,skill_awal,alasan_rekomendasi
0,1,PRG003,Sains Data,S1,Komputer,Data Science dan Analitik,0.3771,0.3214,0.3576,35.76,"analyst, data, grafik, matematika, statistik, suka","Data Analyst, Data Scientist","Dasar Pemrograman Python, Dasar Basis Data dan SQL, Analisis Data Dasar, Statistika Dasar","Cocok karena profil user memiliki kesesuaian pada kata kunci: analyst, data, grafik, matematika, statistik, suka"
1,2,PRG004,Statistika,S1,Matematika dan Sains,Statistika Terapan,0.3359,0.2548,0.3075,30.75,"analyst, data, matematika, statistik","Data Analyst, Data Scientist","Analisis Data Dasar, Statistika Dasar","Cocok karena profil user memiliki kesesuaian pada kata kunci: analyst, data, matematika, statistik"
2,3,PRG002,Sistem Informasi,S1,Komputer,Sistem Bisnis dan Teknologi Informasi,0.2851,0.1714,0.2453,24.53,"analyst, data, matematika","Business Analyst, Data Analyst","Dasar Basis Data dan SQL, Analisis Data Dasar, Komunikasi dan Analisis Kebutuhan","Cocok karena profil user memiliki kesesuaian pada kata kunci: analyst, data, matematika"



ROADMAP BELAJAR UNTUK: Sains Data (PRG003)


,program_id,fase,urutan_fase,durasi_rekomendasi,tujuan_fase,materi_pokok,aktivitas_praktik,output_portofolio,tools_umum,indikator_capaian
0,PRG003,Fondasi,1,1-2 bulan,"Memahami dasar Python, statistik, dan pengolahan data.",Python dasar|Pandas|statistika deskriptif,membersihkan dataset sederhana dan membuat grafik,notebook EDA sederhana,Python|Jupyter|Pandas,Mampu melakukan EDA sederhana.
1,PRG003,Penguatan,2,2-3 bulan,Memahami machine learning dasar dan evaluasi model.,supervised learning|train test split|accuracy|F1-score,membuat model klasifikasi sederhana,notebook model klasifikasi,Scikit-learn|Jupyter,Mampu melatih dan mengevaluasi model dasar.


In [12]:
# ============================================================
# Uji Coba Rekomendasi 2
# ============================================================

test_recommendation_2 = recommend_programs(
    input_text="Aku senang ngoding, membuat aplikasi, dan ingin jadi software engineer",
    minat="coding|logika|komputer|aplikasi",
    mapel="Matematika|Informatika",
    hobi="ngoding|membuat aplikasi",
    gaya_belajar="praktik|logis",
    tujuan_karier="software engineer",
    top_n=3
)

rec_result_2, roadmap_result_2 = show_recommendation_with_roadmap(test_recommendation_2)

TOP REKOMENDASI PROGRAM STUDI


,rank,program_id,nama_program_studi,jenjang,rumpun_ilmu,bidang_keilmuan,text_similarity,structured_score,final_score,final_score_percent,matched_keywords,prospek_karier,skill_awal,alasan_rekomendasi
0,1,PRG001,Teknik Informatika,S1,Komputer,Software Engineering dan Komputasi,0.5661,0.5429,0.5580,55.80,"aplikasi, engineer, pemrograman, software",Software Engineer,"Dasar Pemrograman Python, Dasar Basis Data dan SQL, Algoritma dan Struktur Data Dasar","Cocok karena profil user memiliki kesesuaian pada kata kunci: aplikasi, engineer, pemrograman, software"
1,2,PRG003,Sains Data,S1,Komputer,Data Science dan Analitik,0.0219,0.0500,0.0317,3.17,pemrograman,"Data Analyst, Data Scientist","Dasar Pemrograman Python, Dasar Basis Data dan SQL, Analisis Data Dasar, Statistika Dasar",Cocok karena profil user memiliki kesesuaian pada kata kunci: pemrograman
2,3,PRG002,Sistem Informasi,S1,Komputer,Sistem Bisnis dan Teknologi Informasi,0.0256,0.0357,0.0291,2.91,jadi,"Business Analyst, Data Analyst","Dasar Basis Data dan SQL, Analisis Data Dasar, Komunikasi dan Analisis Kebutuhan",Cocok karena profil user memiliki kesesuaian pada kata kunci: jadi



ROADMAP BELAJAR UNTUK: Teknik Informatika (PRG001)


,program_id,fase,urutan_fase,durasi_rekomendasi,tujuan_fase,materi_pokok,aktivitas_praktik,output_portofolio,tools_umum,indikator_capaian
0,PRG001,Fondasi,1,1-2 bulan,Memahami logika dan dasar pemrograman.,logika algoritma|Python dasar|Git dasar,membuat program kalkulator sederhana|latihan percabangan dan perulangan,repo GitHub berisi latihan Python dasar,Python|VS Code|GitHub,Mampu membuat program sederhana dan menjelaskan alurnya.
1,PRG001,Penguatan,2,2-3 bulan,Memahami basis data dan pengembangan aplikasi sederhana.,SQL dasar|CRUD|HTML CSS dasar|API sederhana,membuat aplikasi catatan sederhana dengan database,aplikasi CRUD sederhana,SQLite|PostgreSQL|Flask/FastAPI,Mampu membuat aplikasi kecil berbasis database.


In [13]:
# ============================================================
# Uji Coba Rekomendasi 3
# Input nonformal / campuran bahasa daerah umum
# ============================================================

test_recommendation_3 = recommend_programs(
    input_text="Aku pengin dadi analis data, seneng angka, statistik, lan olah data",
    minat="data|analisis|statistik",
    mapel="Matematika",
    hobi="olah data|membuat grafik",
    gaya_belajar="analitis",
    tujuan_karier="data analyst",
    top_n=3
)

rec_result_3, roadmap_result_3 = show_recommendation_with_roadmap(test_recommendation_3)

TOP REKOMENDASI PROGRAM STUDI


,rank,program_id,nama_program_studi,jenjang,rumpun_ilmu,bidang_keilmuan,text_similarity,structured_score,final_score,final_score_percent,matched_keywords,prospek_karier,skill_awal,alasan_rekomendasi
0,1,PRG004,Statistika,S1,Matematika dan Sains,Statistika Terapan,0.3459,0.5104,0.4035,40.35,"analis, analitis, analyst, angka, data, matematika, olah, statistik","Data Analyst, Data Scientist","Analisis Data Dasar, Statistika Dasar","Cocok karena profil user memiliki kesesuaian pada kata kunci: analis, analitis, analyst, angka, data, matematika, ol..."
1,2,PRG003,Sains Data,S1,Komputer,Data Science dan Analitik,0.3484,0.3438,0.3468,34.68,"analis, analitis, analyst, angka, data, grafik, matematika, statistik, suka","Data Analyst, Data Scientist","Dasar Pemrograman Python, Dasar Basis Data dan SQL, Analisis Data Dasar, Statistika Dasar","Cocok karena profil user memiliki kesesuaian pada kata kunci: analis, analitis, analyst, angka, data, grafik, matema..."
2,3,PRG002,Sistem Informasi,S1,Komputer,Sistem Bisnis dan Teknologi Informasi,0.2141,0.0938,0.1720,17.20,"analis, analitis, analyst, data, matematika","Business Analyst, Data Analyst","Dasar Basis Data dan SQL, Analisis Data Dasar, Komunikasi dan Analisis Kebutuhan","Cocok karena profil user memiliki kesesuaian pada kata kunci: analis, analitis, analyst, data, matematika"



ROADMAP BELAJAR UNTUK: Statistika (PRG004)


,program_id,fase,urutan_fase,durasi_rekomendasi,tujuan_fase,materi_pokok,aktivitas_praktik,output_portofolio,tools_umum,indikator_capaian


In [14]:
# ============================================================
# Simpan Output Tahap 05
# ============================================================

# Simpan program profile yang sudah diperkaya mapping karier dan skill
program_profile_output_path = DATA_PROCESSED_DIR / "program_recommender_profiles_stage05.csv"
program_df.to_csv(program_profile_output_path, index=False, encoding="utf-8-sig")

# Simpan contoh hasil rekomendasi
sample_recommendations_path = REPORTS_DIR / "sample_recommendations_stage05.csv"
all_sample_recommendations = pd.concat([
    rec_result_1.assign(test_case="Test Case 1 - Data Analyst"),
    rec_result_2.assign(test_case="Test Case 2 - Software Engineer"),
    rec_result_3.assign(test_case="Test Case 3 - Bahasa Nonformal/Daerah")
], ignore_index=True)

all_sample_recommendations.to_csv(sample_recommendations_path, index=False, encoding="utf-8-sig")

# Simpan contoh roadmap top recommendation
roadmap_mapping_path = REPORTS_DIR / "roadmap_mapping_sample_stage05.csv"
all_sample_roadmaps = pd.concat([
    roadmap_result_1.assign(test_case="Test Case 1 - Data Analyst"),
    roadmap_result_2.assign(test_case="Test Case 2 - Software Engineer"),
    roadmap_result_3.assign(test_case="Test Case 3 - Bahasa Nonformal/Daerah")
], ignore_index=True)

all_sample_roadmaps.to_csv(roadmap_mapping_path, index=False, encoding="utf-8-sig")

print("Output Tahap 05 berhasil disimpan:")
print("1.", program_profile_output_path)
print("2.", sample_recommendations_path)
print("3.", roadmap_mapping_path)

Output Tahap 05 berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\data\processed\program_recommender_profiles_stage05.csv
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage05\sample_recommendations_stage05.csv
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\reports\stage05\roadmap_mapping_sample_stage05.csv


In [15]:
# ============================================================
# Simpan Artifact Model Recommender
# ============================================================

tfidf_vectorizer_path = MODELS_DIR / "program_recommender_tfidf_vectorizer.joblib"
program_matrix_path = MODELS_DIR / "program_recommender_matrix.joblib"
recommender_config_path = MODELS_DIR / "program_recommender_config_stage05.json"

joblib.dump(tfidf_vectorizer, tfidf_vectorizer_path)
joblib.dump(program_matrix, program_matrix_path)

recommender_config = {
    "project": "EduPath Career Assistant",
    "stage": "Tahap 05 - Program Studi Recommender System & Roadmap Mapping",
    "method": "Content-Based Filtering using TF-IDF, Cosine Similarity, and Rule-Based Structured Scoring",
    "text_similarity_weight": 0.65,
    "structured_score_weight": 0.35,
    "program_dataset": str(program_path),
    "career_dataset": str(career_path),
    "skill_dataset": str(skill_path),
    "roadmap_dataset": str(roadmap_path),
    "outputs": {
        "program_recommender_profiles": str(program_profile_output_path),
        "sample_recommendations": str(sample_recommendations_path),
        "roadmap_mapping_sample": str(roadmap_mapping_path)
    },
    "notes": (
        "Recommender system ini merupakan baseline content-based recommender. "
        "Dataset masih kecil sehingga hasil rekomendasi perlu divalidasi secara kualitatif "
        "dan akan lebih baik jika dataset diperluas pada tahap berikutnya."
    )
}

with open(recommender_config_path, "w", encoding="utf-8") as file:
    json.dump(recommender_config, file, indent=4, ensure_ascii=False)

print("Artifact model recommender berhasil disimpan:")
print("1.", tfidf_vectorizer_path)
print("2.", program_matrix_path)
print("3.", recommender_config_path)

Artifact model recommender berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\models\program_recommender_tfidf_vectorizer.joblib
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\models\program_recommender_matrix.joblib
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\TUGAS\UAS\edupath-career-assitant\models\program_recommender_config_stage05.json


In [16]:
# ============================================================
# Membuat Summary Report Tahap 05
# ============================================================

summary_text = f"""
TAHAP 05 — Program Studi Recommender System & Roadmap Mapping

Project:
EduPath Career Assistant

Tujuan:
Membangun sistem rekomendasi program studi S1 berbasis profil teks, minat, mapel favorit,
hobi, gaya belajar, dan tujuan karier user.

Metode:
1. Content-Based Filtering
2. TF-IDF Vectorization
3. Cosine Similarity
4. Rule-Based Structured Scoring
5. Roadmap Mapping
6. Career Mapping
7. Skill Mapping

Formula Skor:
Final Score = 0.65 * Text Similarity + 0.35 * Structured Score

Dataset yang digunakan:
1. intent_dataset_processed.csv       : {intent_df.shape[0]} baris
2. program_studi_s1_processed.csv     : {program_df.shape[0]} baris
3. karier_processed.csv               : {career_df.shape[0]} baris
4. skill_awal_processed.csv           : {skill_df.shape[0]} baris
5. roadmap_belajar_processed.csv      : {roadmap_df.shape[0]} baris
6. normalisasi_bahasa_processed.csv   : {normalisasi_df.shape[0]} baris

Output:
1. data/processed/program_recommender_profiles_stage05.csv
2. reports/stage05/sample_recommendations_stage05.csv
3. reports/stage05/roadmap_mapping_sample_stage05.csv
4. models/program_recommender_tfidf_vectorizer.joblib
5. models/program_recommender_matrix.joblib
6. models/program_recommender_config_stage05.json

Catatan Evaluasi:
Sistem recommender ini sudah dapat menghasilkan Top-N program studi, alasan rekomendasi,
prospek karier, skill awal, dan roadmap belajar. Namun, karena dataset masih kecil,
hasil rekomendasi masih bersifat baseline dan perlu diperkuat dengan perluasan dataset
serta validasi domain expert.
"""

summary_path = REPORTS_DIR / "stage05_summary.txt"

with open(summary_path, "w", encoding="utf-8") as file:
    file.write(summary_text)

print(summary_text)
print("Summary report berhasil disimpan di:", summary_path)


TAHAP 05 — Program Studi Recommender System & Roadmap Mapping

Project:
EduPath Career Assistant

Tujuan:
Membangun sistem rekomendasi program studi S1 berbasis profil teks, minat, mapel favorit,
hobi, gaya belajar, dan tujuan karier user.

Metode:
1. Content-Based Filtering
2. TF-IDF Vectorization
3. Cosine Similarity
4. Rule-Based Structured Scoring
5. Roadmap Mapping
6. Career Mapping
7. Skill Mapping

Formula Skor:
Final Score = 0.65 * Text Similarity + 0.35 * Structured Score

Dataset yang digunakan:
1. intent_dataset_processed.csv       : 12 baris
2. program_studi_s1_processed.csv     : 5 baris
3. karier_processed.csv               : 5 baris
4. skill_awal_processed.csv           : 7 baris
5. roadmap_belajar_processed.csv      : 6 baris
6. normalisasi_bahasa_processed.csv   : 18 baris

Output:
1. data/processed/program_recommender_profiles_stage05.csv
2. reports/stage05/sample_recommendations_stage05.csv
3. reports/stage05/roadmap_mapping_sample_stage05.csv
4. models/program_reco

## Interpretasi Hasil

Berdasarkan hasil pengujian, sistem recommender sudah mampu menghasilkan rekomendasi program studi berdasarkan profil input user. Contoh input user yang mengarah pada minat data, statistik, matematika, dan tujuan karier sebagai data analyst cenderung menghasilkan rekomendasi program studi seperti Sains Data atau program lain yang relevan dengan data dan analitik.

Hasil rekomendasi tidak hanya menampilkan nama program studi, tetapi juga:

1. skor kemiripan teks,
2. skor berbasis atribut profil user,
3. skor akhir,
4. kata kunci yang cocok,
5. prospek karier,
6. skill awal yang perlu dipelajari,
7. roadmap belajar.

Dengan demikian, sistem tidak hanya memberikan jawaban berupa nama program studi, tetapi juga memberikan penjelasan yang lebih informatif dan edukatif kepada siswa SMA/SMK/MA.

Namun, hasil pada tahap ini masih dikategorikan sebagai baseline karena jumlah dataset program studi, karier, skill, dan roadmap masih terbatas. Untuk pengembangan berikutnya, dataset perlu diperluas agar sistem dapat memberikan rekomendasi yang lebih beragam dan representatif.

## Kesimpulan Tahap 05

Tahap 05 berhasil membangun baseline Program Studi Recommender System untuk EduPath Career Assistant.

Sistem ini menggunakan pendekatan content-based filtering dengan TF-IDF dan cosine similarity, kemudian diperkuat dengan rule-based structured scoring berdasarkan minat, mata pelajaran favorit, hobi, gaya belajar, dan tujuan karier.

Output utama dari tahap ini adalah rekomendasi Top-N program studi, alasan rekomendasi, mapping prospek karier, mapping skill awal, dan roadmap belajar.

Tahap ini menjadi fondasi penting untuk tahap berikutnya, yaitu integrasi intent classification dan recommender system ke dalam chatbot pipeline.